<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 1.4: Costo Computacional, Precisión Numérica y Condicionamiento

#### Objetivos

- Medir el tiempo de ejecución de operaciones de álgebra lineal y estimar empíricamente su complejidad, contrastándola con las cotas teóricas $O(n)$, $O(n^2)$ y $O(n^3)$.
- Cuantificar la aceleración que produce la vectorización frente a los bucles de Python, y explicar de dónde sale esa diferencia.
- Describir la aritmética de punto flotante IEEE 754: la retícula de valores representables, el epsilon de máquina y la precisión relativa de cada formato.
- Reconocer absorción, cancelación catastrófica, overflow y underflow en un cálculo, y reformular la expresión para evitarlos.
- Distinguir el condicionamiento de un problema de la estabilidad de un algoritmo, y usar el número de condición para acotar la amplificación del error.

## Introducción

Un algoritmo correcto sobre el papel puede resultar inservible en la práctica por dos razones distintas. La primera es el costo: una operación que escala como $n^3$ pasa de milisegundos a horas cuando $n$ se multiplica por cien, y la diferencia entre resolver un sistema lineal con una eliminación gaussiana o invirtiendo la matriz se mide en factores, no en porcentajes. La segunda es la precisión. Las computadoras no operan con números reales sino con un subconjunto finito de racionales, la retícula de punto flotante, y cada operación aritmética introduce un error relativo del orden de $10^{-16}$. Ese error es diminuto, pero existen cálculos que lo amplifican hasta borrar todos los dígitos significativos del resultado.

Este laboratorio separa dos nociones que suelen confundirse. El **condicionamiento** es una propiedad del problema: mide cuánto puede cambiar la respuesta ante una perturbación de los datos, y ningún algoritmo puede esquivarlo. La **estabilidad** es una propiedad del algoritmo: mide cuánto error agrega por encima de lo que el problema ya impone. En las tres partes que siguen vamos a medir tiempos y ajustar exponentes de complejidad, a inspeccionar la representación binaria de los números y sus límites, y a construir casos donde un mismo problema resuelto por dos algoritmos distintos entrega resultados con precisiones que difieren en ocho órdenes de magnitud.

In [ ]:
# Librerías necesarias
import timeit
from decimal import Decimal, getcontext

import numpy as np
import matplotlib.pyplot as plt

getcontext().prec = 50          # 50 dígitos decimales para los cálculos de referencia
eps64 = np.finfo(np.float64).eps  # epsilon de máquina de la doble precisión


def medir_tiempo(f, repeticiones=5):
    """
    Mide el tiempo promedio de una ejecución de f(), en segundos.

    Args:
        f: función sin argumentos (típicamente una lambda que envuelve la operación a medir)
        repeticiones: cantidad de ejecuciones sobre las que se promedia

    Returns:
        t: tiempo promedio por ejecución, en segundos
    """
    return timeit.timeit(f, number=repeticiones) / repeticiones


def exponente_empirico(tamanos, tiempos):
    """
    Estima el exponente p del modelo t(n) = C n^p ajustando una recta por mínimos
    cuadrados a los datos en escala logarítmica: log t = log C + p log n.

    Esta función viene dada y se usa como test numérico de la complejidad de un
    algoritmo: el exponente ajustado debe aproximarse al de la cota teórica.
    """
    p, _ = np.polyfit(np.log(np.asarray(tamanos, dtype=float)),
                      np.log(np.asarray(tiempos, dtype=float)), 1)
    return p


def error_relativo(aproximado, exacto):
    """
    Error relativo |aproximado - exacto| / |exacto|, elemento a elemento.

    Esta función viene dada y se usa en todo el laboratorio para comparar un
    resultado calculado en punto flotante contra un valor de referencia.
    """
    return np.abs(np.asarray(aproximado) - np.asarray(exacto)) / np.abs(np.asarray(exacto))


## Parte 1: Costo Computacional y Vectorización

El costo de un algoritmo numérico se mide contando sus operaciones aritméticas de punto flotante (*flops*: sumas, restas, productos y divisiones) en función del tamaño $n$ de la entrada. Para las operaciones del álgebra lineal densa el recuento es directo:

| Operación | Flops | Orden |
|---|---|---|
| Suma de dos vectores de $\mathbb{R}^n$ | $n$ | $O(n)$ |
| Producto escalar $x^\top y$ | $2n - 1$ | $O(n)$ |
| Producto matriz-vector $Av$, con $A \in \mathbb{R}^{n\times n}$ | $n(2n-1)$ | $O(n^2)$ |
| Producto matriz-matriz $AB$ | $2n^3 - n^2$ | $O(n^3)$ |
| Eliminación gaussiana sobre $Ax = b$ | $\tfrac{2}{3}n^3 + O(n^2)$ | $O(n^3)$ |
| Inversa $A^{-1}$ | $2n^3 + O(n^2)$ | $O(n^3)$ |

La notación asintótica retiene sólo el término dominante: $t(n) = O(g(n))$ si existen $C > 0$ y $n_0$ tales que $t(n) \le C\,g(n)$ para todo $n \ge n_0$. Descarta las constantes, y por eso dos algoritmos del mismo orden pueden diferir en un factor de tres, como la eliminación gaussiana y la inversión.

**Medir el exponente.** Si el tiempo de ejecución sigue el modelo $t(n) = C\,n^p$, tomar logaritmos lo convierte en una recta:

$$\log t(n) = \log C + p \log n,$$

de modo que el exponente $p$ es la pendiente de los datos graficados en escala log-log. Ajustarla por mínimos cuadrados da una estimación empírica de la complejidad, que es lo que hace la función `exponente_empirico` de la celda de imports. La medición tiene ruido: la carga de la máquina, la gestión de energía del procesador (*throttling*) y la resolución del reloj introducen variación entre ejecuciones. Se mitiga promediando varias repeticiones y descartando los tamaños tan chicos que el costo administrativo de la llamada domine sobre el cálculo.

**Por qué la vectorización acelera.** Cada iteración de un bucle `for` de Python sobre un arreglo paga el despacho de bytecode del intérprete, la creación de un objeto `float` por elemento y una indirección de memoria. Las operaciones de NumPy recorren memoria contigua dentro de bucles ya compilados, sin ninguno de esos costos, y las operaciones de álgebra lineal se delegan a BLAS y LAPACK, bibliotecas que trabajan por bloques para aprovechar la jerarquía de cachés y reparten el trabajo entre varios núcleos. La diferencia práctica es de uno a tres órdenes de magnitud, y no cambia la complejidad: un bucle y una llamada a BLAS que calculan el mismo producto escalar son ambos $O(n)$, con constantes muy distintas.

En esta parte vamos a medir esa diferencia, a estimar los exponentes de tres operaciones con complejidad conocida, y a comparar dos maneras de resolver el mismo sistema lineal.

### Ejercicio L1.4.1: Bucles Frente a Operaciones Vectorizadas

**a)** Implementá `producto_escalar_bucle(x, y)`, que calcula $x^\top y = \sum_{i} x_i y_i$ recorriendo los vectores con un bucle `for` de Python.

**b)** Calculá el producto escalar de dos vectores aleatorios de $n = 10^6$ componentes de tres maneras: con tu bucle, con `np.sum(x * y)` y con el operador matricial `x @ y`. Medí el tiempo de cada una con `medir_tiempo` y calculá los factores de aceleración respecto del bucle.

**c)** OPCIONAL: repetí la medición para $n \in \{10^3, 10^4, 10^5, 10^6\}$ y observá si el factor de aceleración depende del tamaño.

**Nota:** las tres implementaciones tienen la misma complejidad $O(n)$, así que la diferencia que vas a medir está toda en la constante. `np.sum(x * y)` recorre la memoria dos veces y crea un arreglo temporal de $n$ elementos; `x @ y` llama a la rutina `ddot` de BLAS, que hace una sola pasada sin memoria auxiliar. El bucle conviene medirlo con pocas repeticiones (`repeticiones=3`) porque tarda bastante.

In [ ]:
np.random.seed(0)


def producto_escalar_bucle(x, y):
    """
    Calcula el producto escalar de dos vectores recorriéndolos con un bucle de Python.

    Args:
        x, y: vectores de igual longitud (arrays 1D)

    Returns:
        total: la suma de los productos x[i] * y[i] (escalar)
    """
    if len(x) != len(y):
        raise ValueError("Los vectores deben tener la misma longitud")

    total = 0.0
    for i in range(len(x)):
        total = ...  # COMPLETAR: acumular el producto x[i] * y[i]
    return total


# b) Las tres implementaciones sobre el mismo par de vectores
n = 1_000_000
x = np.random.rand(n)
y = np.random.rand(n)

p_bucle = ...  # COMPLETAR: producto escalar con el bucle
p_sum = ...    # COMPLETAR: producto escalar con np.sum(x * y)
p_dot = ...    # COMPLETAR: producto escalar con x @ y

t_bucle = ...  # COMPLETAR: medir_tiempo del bucle, con repeticiones=3
t_sum = ...    # COMPLETAR: medir_tiempo de np.sum(x * y), con repeticiones=50
t_dot = ...    # COMPLETAR: medir_tiempo de x @ y, con repeticiones=50

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.isclose(p_bucle, p_dot, rtol=1e-10), "Las tres implementaciones calculan lo mismo"
assert np.isclose(p_sum, p_dot, rtol=1e-10), "Las tres implementaciones calculan lo mismo"
assert t_bucle > 10 * t_dot, "El bucle de Python debe ser órdenes de magnitud más lento que BLAS"
assert t_dot < t_sum, "x @ y hace una sola pasada; np.sum(x * y) crea un temporal de tamaño n"

print(f"Producto escalar : bucle = {p_bucle:.9f} | np.sum = {p_sum:.9f} | x @ y = {p_dot:.9f}")
print(f"bucle    : {t_bucle:.6e} s")
print(f"np.sum   : {t_sum:.6e} s   → {t_bucle / t_sum:8.1f}x más rápido que el bucle")
print(f"x @ y    : {t_dot:.6e} s   → {t_bucle / t_dot:8.1f}x más rápido que el bucle")


### Ejercicio L1.4.2: Medición Empírica de la Complejidad

**a)** Implementá `tiempos_por_tamano(operacion, generador, tamanos, repeticiones)`, que para cada $n$ de `tamanos` construye los argumentos con `generador(n)` (que devuelve una tupla) y devuelve el arreglo de tiempos promedio de `operacion(*args)`.

**b)** Usala para medir tres operaciones de complejidad teórica conocida y estimá el exponente de cada una con `exponente_empirico`:

- suma de dos vectores de largo $n$, esperado $O(n)$;
- producto matriz-vector con $A \in \mathbb{R}^{n \times n}$, esperado $O(n^2)$;
- producto matriz-matriz con $A, B \in \mathbb{R}^{n \times n}$, esperado $O(n^3)$.

**c)** OPCIONAL: agregá el cálculo del determinante y el de la inversa, ambos $O(n^3)$, y compará sus constantes contra la del producto matriz-matriz.

**Nota:** el generador se pasa como función, y no como datos ya construidos, para que el costo de generar las matrices aleatorias quede fuera de la medición. Cada operación usa su propio rango de tamaños: si $n$ es tan chico que el tiempo medido es comparable al costo de la llamada a la función, el exponente ajustado sale por debajo del teórico. Por la misma razón el producto matriz-matriz suele dar un exponente algo menor que $3$: BLAS aprovecha mejor la caché y los múltiples núcleos a medida que $n$ crece. El bloque de verificación grafica los tiempos en escala log-log junto con la recta ajustada, cuya pendiente es el exponente estimado.

In [ ]:
np.random.seed(1)


def tiempos_por_tamano(operacion, generador, tamanos, repeticiones=5):
    """
    Mide el tiempo de ejecución de una operación para una sucesión de tamaños de entrada.

    Args:
        operacion: función que recibe los argumentos generados y ejecuta la operación a medir
        generador: función que, dado n, devuelve la TUPLA de argumentos para operacion
        tamanos: lista o array con los tamaños n a medir
        repeticiones: repeticiones por tamaño sobre las que se promedia

    Returns:
        tiempos: tiempo promedio de cada tamaño (array del mismo largo que tamanos)
    """
    tiempos = []
    for n in tamanos:
        args = ...  # COMPLETAR: argumentos de tamaño n
        t = ...     # COMPLETAR: medir con medir_tiempo el tiempo de operacion(*args)
        tiempos.append(t)
    return np.array(tiempos)


# b) Tres operaciones con complejidad teórica conocida
tam_vec = np.array([10**4, 10**5, 10**6, 10**7])
tam_matvec = np.array([400, 800, 1600, 3200])
tam_matmat = np.array([200, 400, 800, 1600])

gen_dos_vectores = lambda n: (np.random.rand(n), np.random.rand(n))
gen_matriz_vector = lambda n: (np.random.rand(n, n), np.random.rand(n))
gen_dos_matrices = lambda n: (np.random.rand(n, n), np.random.rand(n, n))

t_suma = ...    # COMPLETAR: tiempos de a + b sobre tam_vec, con repeticiones=20
t_matvec = ...  # COMPLETAR: tiempos de A @ v sobre tam_matvec, con repeticiones=20
t_matmat = ...  # COMPLETAR: tiempos de A @ B sobre tam_matmat, con repeticiones=3

p_suma = ...    # COMPLETAR: exponente empírico de la suma de vectores
p_matvec = ...  # COMPLETAR: exponente empírico del producto matriz-vector
p_matmat = ...  # COMPLETAR: exponente empírico del producto matriz-matriz

# ── Verificación ─────────────────────────────────────────────────────────────
mediciones = [("Suma de vectores", tam_vec, t_suma, p_suma, 1),
              ("Matriz por vector", tam_matvec, t_matvec, p_matvec, 2),
              ("Matriz por matriz", tam_matmat, t_matmat, p_matmat, 3)]

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (titulo, tam, tiempos, p, teorico) in zip(ejes, mediciones):
    ax.loglog(tam, tiempos, "o-", label="medido")
    ajuste = tiempos[-1] * (tam / tam[-1]) ** p
    ax.loglog(tam, ajuste, "--", label=f"ajuste $n^{{{p:.2f}}}$")
    ax.loglog(tam, tiempos[-1] * (tam / tam[-1]) ** teorico, ":", label=f"teórico $n^{teorico}$")
    ax.set_xlabel("tamaño $n$")
    ax.set_ylabel("tiempo (s)")
    ax.set_title(titulo)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

assert 0.6 < p_suma < 1.5, "La suma de vectores es O(n): el exponente debe rondar 1"
assert 1.5 < p_matvec < 2.5, "El producto matriz-vector es O(n^2)"
assert 2.2 < p_matmat < 3.4, "El producto matriz-matriz es O(n^3)"

for titulo, _, _, p, teorico in mediciones:
    print(f"{titulo:20s} → exponente ajustado = {p:.3f} | teórico = {teorico}")


### Ejercicio L1.4.3: Resolver $Ax = b$, Inversa Frente a Eliminación Gaussiana

Los dos caminos para resolver un sistema lineal tienen la misma complejidad $O(n^3)$ pero difieren en la constante y, sobre todo, en la precisión. Calcular $A^{-1}$ y multiplicar por $b$ cuesta unos $2n^3$ flops; la eliminación gaussiana que implementa `np.linalg.solve` cuesta $\tfrac{2}{3}n^3$ y además evita formar la inversa, que es una matriz llena de errores de redondeo cuando $A$ está mal condicionada.

**a)** Implementá `resolver_por_inversa(A, b)` y `resolver_por_gaussiana(A, b)`.

**b)** Medí el tiempo de ambas sobre un sistema bien condicionado de $1000 \times 1000$ y calculá el cociente entre los dos tiempos.

**c)** Sobre un sistema mal condicionado de $300 \times 300$ con $\kappa_2(A) = 10^{12}$ y solución exacta $x = (1,\dots,1)^\top$, calculá el residuo relativo $\|Ax - b\| / \|b\|$ de cada método y el error relativo respecto de la solución exacta.

**Nota:** el andamiaje incluye la función `matriz_con_condicion`, que construye una matriz con número de condición prefijado a partir de dos matrices ortogonales y una lista de valores singulares. Prestá atención a la distinción entre **residuo** (cuánto le falta a $x$ para satisfacer la ecuación) y **error** (cuánto se aleja $x$ de la solución verdadera): son cosas distintas, y el número de condición es justamente el factor que las relaciona.

In [ ]:
np.random.seed(2)


def matriz_con_condicion(n, kappa, semilla=0):
    """
    Genera una matriz n x n con número de condición 2 igual a kappa.

    Esta función viene dada: construye A = U diag(s) V^T con U y V ortogonales
    y valores singulares s espaciados logarítmicamente entre 1 y 1/kappa.
    """
    rng = np.random.default_rng(semilla)
    U, _ = np.linalg.qr(rng.standard_normal((n, n)))
    V, _ = np.linalg.qr(rng.standard_normal((n, n)))
    s = np.logspace(0.0, -np.log10(kappa), n)
    return (U * s) @ V.T


def resolver_por_inversa(A, b):
    """
    Resuelve A x = b calculando explícitamente la inversa de A.

    Args:
        A: matriz cuadrada no singular (n x n)
        b: término independiente (array 1D de largo n)

    Returns:
        x: solución del sistema (array 1D de largo n)
    """
    return ...  # COMPLETAR: usar np.linalg.inv


def resolver_por_gaussiana(A, b):
    """
    Resuelve A x = b por eliminación gaussiana con pivoteo parcial.

    Args:
        A: matriz cuadrada no singular (n x n)
        b: término independiente (array 1D de largo n)

    Returns:
        x: solución del sistema (array 1D de largo n)
    """
    return ...  # COMPLETAR: usar np.linalg.solve


# b) Costo sobre un sistema bien condicionado de 1000 x 1000
n = 1000
A_bien = matriz_con_condicion(n, 10.0, semilla=1)
b_bien = np.random.rand(n)

t_inv = ...    # COMPLETAR: medir_tiempo de resolver_por_inversa(A_bien, b_bien), repeticiones=5
t_gauss = ...  # COMPLETAR: medir_tiempo de resolver_por_gaussiana(A_bien, b_bien), repeticiones=5

# c) Precisión sobre un sistema mal condicionado de 300 x 300
A_mal = matriz_con_condicion(300, 1e12, semilla=2)
x_exacto = np.ones(300)
b_mal = A_mal @ x_exacto

x_inv = ...    # COMPLETAR: solución por inversa
x_gauss = ...  # COMPLETAR: solución por eliminación gaussiana

res_inv = ...    # COMPLETAR: residuo relativo ||A x_inv - b|| / ||b||
res_gauss = ...  # COMPLETAR: residuo relativo ||A x_gauss - b|| / ||b||

err_inv = ...    # COMPLETAR: error relativo ||x_inv - x_exacto|| / ||x_exacto||
err_gauss = ...  # COMPLETAR: error relativo ||x_gauss - x_exacto|| / ||x_exacto||

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.allclose(resolver_por_inversa(A_bien, b_bien),
                   resolver_por_gaussiana(A_bien, b_bien)), "Ambos métodos resuelven el mismo sistema"
assert t_inv > t_gauss, "Invertir cuesta unos 2n^3 flops frente a los 2n^3/3 de la eliminación"
assert res_gauss < 1e-13, "La eliminación gaussiana es estable hacia atrás: el residuo es del orden de eps"
assert res_inv > res_gauss, "Formar la inversa agrega errores de redondeo que se ven en el residuo"

print(f"n = {n}, sistema bien condicionado")
print(f"  inversa    : {t_inv:.4f} s")
print(f"  gaussiana  : {t_gauss:.4f} s   → cociente {t_inv / t_gauss:.2f}x")
print(f"\nn = 300, kappa_2(A) = {np.linalg.cond(A_mal):.2e}")
print(f"  residuo relativo : inversa = {res_inv:.3e} | gaussiana = {res_gauss:.3e}")
print(f"  error relativo   : inversa = {err_inv:.3e} | gaussiana = {err_gauss:.3e}")
print(f"  cota kappa * eps : {np.linalg.cond(A_mal) * eps64:.3e}")


## Parte 2: Aritmética de Punto Flotante

Una computadora no almacena números reales sino un subconjunto finito de racionales. El estándar IEEE 754 representa cada número normalizado como

$$x = \pm\, (1.b_1 b_2 \dots b_{t-1})_2 \times 2^{e},$$

con $t$ bits de mantisa (contando el $1$ implícito) y un exponente $e$ acotado. Los tres formatos que usa NumPy son:

| Formato | Bits (signo/exponente/mantisa) | $t$ | $\varepsilon = 2^{-(t-1)}$ | Rango aproximado |
|---|---|---|---|---|
| `float16` | 1 / 5 / 10 | 11 | $9.8 \times 10^{-4}$ | $10^{\pm 5}$ |
| `float32` | 1 / 8 / 23 | 24 | $1.2 \times 10^{-7}$ | $10^{\pm 38}$ |
| `float64` | 1 / 11 / 52 | 53 | $2.2 \times 10^{-16}$ | $10^{\pm 308}$ |

**El epsilon de máquina.** La cantidad $\varepsilon = 2^{-(t-1)}$ es la distancia entre $1$ y el flotante inmediatamente superior. Entre dos potencias de dos consecutivas, $[2^e, 2^{e+1})$, los flotantes están espaciados uniformemente a distancia $2^{e}\varepsilon$, de manera que el **espaciado relativo** de la retícula es siempre del orden de $\varepsilon$ y el espaciado absoluto crece con la magnitud. Redondear un real $x$ al flotante más cercano da entonces

$$\mathrm{fl}(x) = x(1 + \delta), \qquad |\delta| \le u = \tfrac{\varepsilon}{2},$$

donde $u$ es la **unidad de redondeo**. El estándar exige que cada operación aritmética se calcule como si se hiciera en precisión infinita y luego se redondeara, de modo que

$$x \oplus y = (x + y)(1 + \delta), \qquad |\delta| \le u,$$

y lo mismo para resta, producto y cociente. Cada operación individual es entonces casi perfecta, pero eso no impide que una secuencia de operaciones dé un resultado sin ningún dígito correcto. Tres mecanismos lo explican.

**Absorción.** Si $|y| < u\,|x|$, el resultado exacto $x + y$ cae más cerca de $x$ que de cualquier otro flotante, y la suma devuelve $x$. El sumando chico desaparece. Sumar $10^{6}$ términos de tamaño $1$ a un acumulador que ya vale $10^{16}$ no cambia nada.

**Cancelación catastrófica.** Restar dos números cercanos es una operación correctamente redondeada, pero destructiva cuando los operandos ya traían error. Si $\hat{x} = x(1+\delta_x)$ e $\hat{y} = y(1+\delta_y)$, el error relativo de la resta cumple

$$\frac{|(\hat{x} - \hat{y}) - (x - y)|}{|x - y|} \le \frac{|x| + |y|}{|x - y|} \max(|\delta_x|, |\delta_y|).$$

El factor $\frac{|x| + |y|}{|x-y|}$ es el número de condición de la resta y explota cuando $x \approx y$. No se pierde información en la resta: se pierde en el hecho de que los dígitos que sobreviven eran los menos confiables de los operandos.

**Overflow y underflow.** Fuera del rango del exponente no hay representación. En `float64` un resultado mayor que $\approx 1.8 \times 10^{308}$ se convierte en `inf` y uno menor que $\approx 5 \times 10^{-324}$ colapsa a cero. Aparece al exponenciar: $e^{800} \approx 10^{347}$ desborda, aunque el cociente $e^{800}/e^{799}$ que uno quería calcular valga $e$. La salida habitual es trabajar con logaritmos y restar el máximo antes de exponenciar.

### Ejercicio L1.4.4: La Retícula de Punto Flotante y el Epsilon de Máquina

**a)** Armá un resumen de los tres formatos usando `np.finfo`: epsilon de máquina, bits de mantisa y mayor valor representable de `np.float16`, `np.float32` y `np.float64`.

**b)** Implementá `espaciado_relativo(x, dtype)`, que devuelve $\dfrac{\mathrm{next}(x) - x}{x}$, donde $\mathrm{next}(x)$ es el flotante inmediatamente mayor que $x$ en el formato `dtype`. Evaluala en $x = 10^{-3}, 1, 10^{3}$ con `float64` y compará los tres valores contra $\varepsilon$, junto con los espaciados absolutos correspondientes.

**c)** Guardá $0.1$ en los tres formatos e imprimí cada valor con 20 decimales para ver a qué número se redondeó realmente.

**Nota:** `np.nextafter(x, np.inf)` devuelve el flotante siguiente en dirección a $+\infty$. Para comparar contra el valor decimal exacto conviene el módulo `decimal`: `Decimal(x)` construye el racional **exacto** que representa el flotante `x`, mientras que `Decimal(1) / Decimal(10)` es el decimal exacto $0.1$.

In [ ]:
def espaciado_relativo(x, dtype=np.float64):
    """
    Distancia relativa entre x y el flotante inmediatamente mayor, en el formato dtype.

    Args:
        x: valor positivo
        dtype: formato de punto flotante (np.float16, np.float32 o np.float64)

    Returns:
        d: (siguiente(x) - x) / x, como float de Python
    """
    x_dt = dtype(x)
    siguiente = ...  # COMPLETAR: flotante inmediatamente mayor que x_dt, con np.nextafter
    return ...       # COMPLETAR: distancia relativa entre x_dt y siguiente


# a) Resumen de los tres formatos
formatos = [np.float16, np.float32, np.float64]
epsilons = ...      # COMPLETAR: array con np.finfo(f).eps para cada formato
bits_mantisa = ...  # COMPLETAR: lista con np.finfo(f).nmant para cada formato
maximos = ...       # COMPLETAR: lista con np.finfo(f).max para cada formato

# b) Espaciado de la retícula en tres magnitudes distintas
puntos = np.array([1e-3, 1.0, 1e3])
esp_relativo = ...  # COMPLETAR: array con espaciado_relativo(v) para cada v de puntos
esp_absoluto = ...  # COMPLETAR: array con np.nextafter(v, np.inf) - v para cada v de puntos

# c) A qué número se redondea realmente 0.1 en cada formato
valores_01 = ...  # COMPLETAR: lista con float(f(0.1)) para cada formato

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.all(esp_relativo > eps64 / 2) and np.all(esp_relativo <= eps64 * (1 + 1e-12)), \
    "El espaciado relativo de la retícula está siempre entre eps/2 y eps"
assert esp_absoluto[0] < esp_absoluto[1] < esp_absoluto[2], \
    "El espaciado ABSOLUTO crece con la magnitud del número"
assert epsilons[0] > epsilons[1] > epsilons[2], "Más bits de mantisa significan un eps más chico"
assert Decimal(valores_01[2]) != Decimal(1) / Decimal(10), \
    "0.1 no tiene representación binaria finita: ningún formato lo guarda exacto"

for f, e, m, mx in zip(formatos, epsilons, bits_mantisa, maximos):
    print(f"{f.__name__:9s} | eps = {e:.6e} | bits de mantisa = {m:2d} | max = {mx:.3e}")
print()
for v, dr, da in zip(puntos, esp_relativo, esp_absoluto):
    print(f"x = {v:8.3e} → espaciado absoluto = {da:.6e} | relativo = {dr:.6e}")
print(f"\neps de float64 = {eps64:.6e}")
print()
for f, v in zip(formatos, valores_01):
    print(f"0.1 en {f.__name__:9s} = {v:.20f}")


### Ejercicio L1.4.5: Errores de la Aritmética de Punto Flotante

Construí los cuatro casos que ilustran los mecanismos de la introducción. En cada uno hay que calcular el error relativo respecto del valor exacto y compararlo con $\varepsilon$.

**a) Redondeo.** Calculá $z = 0.1 + 0.2$ y evaluá `z == 0.3`. Calculá el error relativo de $z$ respecto de $0.3$.

**b) Absorción.** Calculá $z = 10^{16} + 1$ y evaluá `z == 1e16`. El valor exacto es el entero $10^{16} + 1$, que hay que construir con `Decimal` porque no es representable en `float64`.

**c) Cancelación catastrófica.** Con $x = 0.123456789012345678$ e $y = 0.123456789012345677$, cuya diferencia exacta es $10^{-18}$, calculá $z = x - y$ y el factor de amplificación $\frac{|x| + |y|}{|x - y|}$ usando la diferencia exacta.

**d) Overflow y underflow.** Estimá el orden de magnitud de $e^{800}$ mediante $\log_{10}(e^{800}) = 800 \log_{10} e$, compará con $\log_{10}$ del mayor `float64` representable, y calculá $e^{800}$ y $e^{-800}$ en NumPy.

**Nota:** en los casos b) y c) el valor exacto no es representable en punto flotante, así que se lo construye con `Decimal`; el andamiaje ya lo hace. El bloque `np.errstate(over="ignore", under="ignore")` evita que NumPy emita advertencias al desbordar.

In [ ]:
# a) Redondeo: la suma más famosa
suma = ...      # COMPLETAR: 0.1 + 0.2
err_suma = ...  # COMPLETAR: error relativo de suma respecto de 0.3

# b) Absorción: un sumando que desaparece
grande = 1e16
suma_absorbida = ...   # COMPLETAR: grande + 1.0
exacto_absorcion = Decimal(10) ** 16 + Decimal(1)
err_absorcion = ...    # COMPLETAR: |Decimal(suma_absorbida) - exacto_absorcion| / exacto_absorcion, como float

# c) Cancelación catastrófica
x_c = 0.123456789012345678
y_c = 0.123456789012345677
exacto_resta = float(Decimal("0.123456789012345678") - Decimal("0.123456789012345677"))

resta = ...          # COMPLETAR: x_c - y_c
err_resta = ...      # COMPLETAR: error relativo de resta respecto de exacto_resta
amplificacion = ...  # COMPLETAR: (|x_c| + |y_c|) / exacto_resta

# d) Overflow y underflow
orden_exp800 = ...  # COMPLETAR: 800 * log10(e)
orden_maximo = ...  # COMPLETAR: log10 del mayor float64 representable
with np.errstate(over="ignore", under="ignore"):
    exp_800 = ...      # COMPLETAR: np.exp(800.0)
    exp_menos_800 = ...  # COMPLETAR: np.exp(-800.0)

# ── Verificación ─────────────────────────────────────────────────────────────
assert suma != 0.3, "0.1, 0.2 y 0.3 se redondean, y la suma de los redondeos no es el redondeo de la suma"
assert err_suma < eps64, "Aun así la suma está correctamente redondeada: el error relativo no supera eps"
assert suma_absorbida == grande, "El 1 queda por debajo del espaciado de la retícula en 1e16"
assert err_absorcion < eps64, "La absorción respeta el modelo fl(x+y) = (x+y)(1+delta)"
assert resta == 0.0, "Ambos literales se redondean al MISMO float64: la resta da exactamente cero"
assert err_resta == 1.0, "El error relativo de la cancelación es del 100%"
assert amplificacion > 1e17, "El número de condición de la resta explota cuando los operandos son cercanos"
assert orden_exp800 > orden_maximo and np.isinf(exp_800), "e^800 desborda el rango de float64"
assert exp_menos_800 == 0.0, "e^-800 cae por debajo del menor subnormal y colapsa a cero"

print(f"a) 0.1 + 0.2 = {suma:.20f}   (== 0.3 → {suma == 0.3})")
print(f"   error relativo = {err_suma:.3e}   |   eps = {eps64:.3e}")
print(f"\nb) 1e16 + 1 = {suma_absorbida:.1f}   (== 1e16 → {suma_absorbida == grande})")
print(f"   error relativo = {err_absorcion:.3e}")
print(f"\nc) x - y = {resta:.3e}   (exacto = {exacto_resta:.3e})")
print(f"   error relativo = {err_resta:.3e}   |   amplificación = {amplificacion:.3e}")
print(f"\nd) log10(e^800) = {orden_exp800:.2f}   |   log10(max float64) = {orden_maximo:.2f}")
print(f"   e^800 = {exp_800}   |   e^-800 = {exp_menos_800}")


## Conclusiones

En este laboratorio hemos explorado:

1. **Costo y vectorización**: un bucle de Python y una llamada a BLAS que calculan el mismo producto escalar tienen la misma complejidad $O(n)$ y difieren en dos o tres órdenes de magnitud de tiempo. La complejidad describe cómo escala el algoritmo; la constante decide si el cálculo tarda un segundo o una hora, y esa constante depende de evitar el intérprete y usar rutinas compiladas sobre memoria contigua.
2. **Medición empírica de la complejidad**: graficados en escala log-log, los tiempos de una operación $O(n^p)$ se alinean sobre una recta de pendiente $p$. Ajustarla recupera el exponente teórico de la suma de vectores, el producto matriz-vector y el producto matriz-matriz, siempre que se descarten los tamaños donde el costo de la llamada domina sobre el cálculo.
3. **Aritmética de punto flotante**: la retícula de valores representables tiene espaciado relativo constante del orden de $\varepsilon$, y cada operación aritmética está correctamente redondeada. Eso no alcanza para que una secuencia de operaciones sea confiable: la absorción hace desaparecer los sumandos chicos, la cancelación catastrófica amplifica el error preexistente por el factor $\frac{|x|+|y|}{|x-y|}$, y fuera del rango del exponente aparecen `inf` y $0$.

## Declaración de uso de inteligencia artificial

> **Política del curso (syllabus).** Se permite usar herramientas de IA (ChatGPT, Copilot, Claude, etc.)
> *como apoyo para el aprendizaje*: entender conceptos, explorar ideas, depurar código o buscar
> explicaciones alternativas. **No** se permite usarlas para **resolver los ejercicios evaluados** ni
> para **verificar las respuestas antes de entregar**. Se espera que cada estudiante resuelva todos los
> problemas por sí mismo/a.

Completá esta declaración **escribiendo tu respuesta** donde aparece «…» (doble clic en esta celda para
editarla y luego Ctrl/Cmd + Enter para volver a renderizarla):

**1. ¿Usaste herramientas de IA en este laboratorio?** (Sí / No): «…»

**2. ¿Cuál(es)?** (ChatGPT, Copilot, Claude, …; escribí "ninguna" si no usaste): «…»

**3. ¿Para qué la(s) usaste?** Usos permitidos: entender conceptos, explorar ideas, depurar código,
buscar explicaciones alternativas. Escribí los que apliquen: «…»

**4. Detalle breve:** «En qué ejercicios y de qué manera. Ej.: "Usé Claude para entender la cancelación
catastrófica antes del ejercicio L1.4.5."»

**Declaración de honestidad académica.** Declaro que resolví los ejercicios de este laboratorio por mí
mismo/a y que no utilicé herramientas de IA para resolver los ejercicios evaluados ni para verificar mis
respuestas antes de entregar, de acuerdo con la política del curso.

**Nombre y apellido:** «…»          **Fecha:** «…»